# NP-完全问题规约到 SAT

## 从具体问题到布尔可满足性

本 notebook 展示如何将各种 NP-完全问题**规约**到 SAT（布尔可满足性问题）。

这是理解 NP-完全性的关键：**所有 NP 问题都可以在多项式时间内规约到 SAT**。

### 内容
1. SAT 问题简介
2. 图着色 (Graph Coloring) → SAT
3. 团问题 (Clique) → SAT
4. 顶点覆盖 (Vertex Cover) → SAT
5. 哈密顿路径 (Hamiltonian Path) → SAT
6. 子集和 (Subset Sum) → SAT

---

In [16]:
!pip install python-sat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 22.6 MB/s eta 0:00:00


In [17]:
import numpy as np
from itertools import combinations
from typing import List, Tuple, Set, Dict

# 尝试导入 SAT solver
try:
    from pysat.solvers import Glucose3
    from pysat.formula import CNF
    HAS_PYSAT = True
    print("✓ PySAT 可用，可以验证 SAT 公式")
except ImportError:
    HAS_PYSAT = False
    print("✗ PySAT 未安装，将只生成 CNF 公式而不求解")
    print("  安装方法: pip install python-sat")

✓ PySAT 可用，可以验证 SAT 公式


---
## 1. SAT 问题简介

### 1.1 什么是 SAT？

**SAT（Boolean Satisfiability Problem）**：给定一个布尔公式，判断是否存在一组变量赋值使其为真。

**CNF（合取范式）形式**：
$$\phi = (x_1 \lor \neg x_2 \lor x_3) \land (\neg x_1 \lor x_2) \land (x_2 \lor x_3)$$

- **文字（Literal）**：变量 $x$ 或其否定 $\neg x$
- **子句（Clause）**：文字的析取（OR）
- **CNF**：子句的合取（AND）

### 1.2 DIMACS CNF 格式

标准的 SAT 问题表示格式：
```
p cnf <变量数> <子句数>
<子句1>
<子句2>
...
```

其中正数表示变量，负数表示变量的否定，0 表示子句结束。

In [18]:
def cnf_to_dimacs(clauses: List[List[int]], num_vars: int = None) -> str:
    """将子句列表转换为 DIMACS CNF 格式"""
    if num_vars is None:
        num_vars = max(abs(lit) for clause in clauses for lit in clause)

    lines = [f"p cnf {num_vars} {len(clauses)}"]
    for clause in clauses:
        lines.append(" ".join(map(str, clause)) + " 0")
    return "\n".join(lines)

def solve_sat(clauses: List[List[int]]) -> Tuple[bool, List[int]]:
    """求解 SAT 问题"""
    if not HAS_PYSAT:
        return None, None

    solver = Glucose3()
    for clause in clauses:
        solver.add_clause(clause)

    satisfiable = solver.solve()
    model = solver.get_model() if satisfiable else None
    solver.delete()

    return satisfiable, model

def print_cnf_info(clauses: List[List[int]], problem_name: str):
    """打印 CNF 公式信息"""
    num_vars = max(abs(lit) for clause in clauses for lit in clause) if clauses else 0
    print(f"\n{'='*60}")
    print(f"{problem_name}")
    print(f"{'='*60}")
    print(f"变量数: {num_vars}")
    print(f"子句数: {len(clauses)}")
    print(f"\nDIMACS CNF 格式:")
    print("-" * 40)
    print(cnf_to_dimacs(clauses, num_vars))

---
## 2. 图着色问题 → SAT

### 2.1 问题定义

**k-图着色问题**：给定图 $G = (V, E)$，判断是否能用 $k$ 种颜色给所有顶点着色，使得相邻顶点颜色不同。

### 2.2 规约方法

**变量定义**：$x_{v,c}$ 表示「顶点 $v$ 被涂颜色 $c$」

**约束条件**：

1. **每个顶点至少有一种颜色**：
   $$\forall v: (x_{v,1} \lor x_{v,2} \lor \cdots \lor x_{v,k})$$

2. **每个顶点最多有一种颜色**（可选，加强约束）：
   $$\forall v, \forall c_1 < c_2: (\neg x_{v,c_1} \lor \neg x_{v,c_2})$$

3. **相邻顶点颜色不同**：
   $$\forall (u,v) \in E, \forall c: (\neg x_{u,c} \lor \neg x_{v,c})$$

In [19]:
def graph_coloring_to_sat(vertices: List[int], edges: List[Tuple[int, int]], k: int) -> Tuple[List[List[int]], Dict]:
    """
    将 k-图着色问题规约到 SAT

    输入:
        vertices: 顶点列表 [0, 1, 2, ...]
        edges: 边列表 [(0,1), (1,2), ...]
        k: 颜色数

    输出:
        clauses: CNF 子句列表
        var_map: 变量映射 {(v, c): var_id}
    """
    clauses = []
    var_map = {}  # (vertex, color) -> variable_id

    # 创建变量映射
    var_id = 1
    for v in vertices:
        for c in range(1, k + 1):
            var_map[(v, c)] = var_id
            var_id += 1

    # 约束1: 每个顶点至少有一种颜色
    for v in vertices:
        clause = [var_map[(v, c)] for c in range(1, k + 1)]
        clauses.append(clause)

    # 约束2: 每个顶点最多有一种颜色 (At-Most-One)
    for v in vertices:
        for c1 in range(1, k + 1):
            for c2 in range(c1 + 1, k + 1):
                clauses.append([-var_map[(v, c1)], -var_map[(v, c2)]])

    # 约束3: 相邻顶点颜色不同
    for (u, v) in edges:
        for c in range(1, k + 1):
            clauses.append([-var_map[(u, c)], -var_map[(v, c)]])

    return clauses, var_map

def decode_coloring(model: List[int], var_map: Dict, vertices: List[int], k: int) -> Dict[int, int]:
    """从 SAT 解码出着色方案"""
    if model is None:
        return None

    model_set = set(model)
    coloring = {}

    for v in vertices:
        for c in range(1, k + 1):
            if var_map[(v, c)] in model_set:
                coloring[v] = c
                break

    return coloring

In [20]:
# 示例：三角形图的 3-着色
print("示例: 三角形图 (3个顶点, 3条边) 的 3-着色问题")
print("\n原问题:")
print("  顶点: {0, 1, 2}")
print("  边: {(0,1), (1,2), (0,2)}")
print("  问题: 能否用 3 种颜色着色，使相邻顶点颜色不同?")

vertices = [0, 1, 2]
edges = [(0, 1), (1, 2), (0, 2)]
k = 3

clauses, var_map = graph_coloring_to_sat(vertices, edges, k)

print("\n变量定义:")
for (v, c), var_id in var_map.items():
    print(f"  x_{var_id} = '顶点{v}涂颜色{c}'")

print_cnf_info(clauses, "生成的 SAT 公式")

# 求解
if HAS_PYSAT:
    sat, model = solve_sat(clauses)
    print(f"\n求解结果: {'可满足' if sat else '不可满足'}")
    if sat:
        coloring = decode_coloring(model, var_map, vertices, k)
        print(f"着色方案: {coloring}")
        print("验证:")
        for (u, v) in edges:
            status = "✓" if coloring[u] != coloring[v] else "✗"
            print(f"  边({u},{v}): 颜色{coloring[u]} vs 颜色{coloring[v]} {status}")

示例: 三角形图 (3个顶点, 3条边) 的 3-着色问题

原问题:
  顶点: {0, 1, 2}
  边: {(0,1), (1,2), (0,2)}
  问题: 能否用 3 种颜色着色，使相邻顶点颜色不同?

变量定义:
  x_1 = '顶点0涂颜色1'
  x_2 = '顶点0涂颜色2'
  x_3 = '顶点0涂颜色3'
  x_4 = '顶点1涂颜色1'
  x_5 = '顶点1涂颜色2'
  x_6 = '顶点1涂颜色3'
  x_7 = '顶点2涂颜色1'
  x_8 = '顶点2涂颜色2'
  x_9 = '顶点2涂颜色3'

生成的 SAT 公式
变量数: 9
子句数: 21

DIMACS CNF 格式:
----------------------------------------
p cnf 9 21
1 2 3 0
4 5 6 0
7 8 9 0
-1 -2 0
-1 -3 0
-2 -3 0
-4 -5 0
-4 -6 0
-5 -6 0
-7 -8 0
-7 -9 0
-8 -9 0
-1 -4 0
-2 -5 0
-3 -6 0
-4 -7 0
-5 -8 0
-6 -9 0
-1 -7 0
-2 -8 0
-3 -9 0

求解结果: 可满足
着色方案: {0: 3, 1: 2, 2: 1}
验证:
  边(0,1): 颜色3 vs 颜色2 ✓
  边(1,2): 颜色2 vs 颜色1 ✓
  边(0,2): 颜色3 vs 颜色1 ✓


In [21]:
# 示例：四顶点完全图的 3-着色（不可能）
print("\n" + "="*60)
print("示例: 四顶点完全图 K4 的 3-着色问题")
print("="*60)
print("\n原问题:")
print("  顶点: {0, 1, 2, 3}")
print("  边: 所有顶点两两相连 (完全图)")
print("  问题: 能否用 3 种颜色着色?")

vertices_k4 = [0, 1, 2, 3]
edges_k4 = [(i, j) for i in range(4) for j in range(i+1, 4)]  # 完全图

clauses_k4, var_map_k4 = graph_coloring_to_sat(vertices_k4, edges_k4, k=3)

print(f"\n生成的 SAT 公式: {len(clauses_k4)} 个子句")

if HAS_PYSAT:
    sat, model = solve_sat(clauses_k4)
    print(f"\n求解结果: {'可满足' if sat else '不可满足'}")
    if not sat:
        print("说明: K4 至少需要 4 种颜色 (色数 = 4)")


示例: 四顶点完全图 K4 的 3-着色问题

原问题:
  顶点: {0, 1, 2, 3}
  边: 所有顶点两两相连 (完全图)
  问题: 能否用 3 种颜色着色?

生成的 SAT 公式: 34 个子句

求解结果: 不可满足
说明: K4 至少需要 4 种颜色 (色数 = 4)


---
## 3. 团问题 (Clique) → SAT

### 3.1 问题定义

**k-团问题**：给定图 $G = (V, E)$，判断是否存在大小为 $k$ 的团（完全子图）。

### 3.2 规约方法

**变量定义**：$x_v$ 表示「顶点 $v$ 在团中」

**约束条件**：

1. **恰好选择 k 个顶点**：使用基数约束（这里简化为至少 k 个）

2. **团中的顶点两两相连**：
   $$\forall u, v \text{ 不相邻}: (\neg x_u \lor \neg x_v)$$
   即：如果 $u, v$ 不相邻，则它们不能同时在团中

In [22]:
def clique_to_sat(vertices: List[int], edges: List[Tuple[int, int]], k: int) -> Tuple[List[List[int]], Dict]:
    """
    将 k-团问题规约到 SAT

    输入:
        vertices: 顶点列表
        edges: 边列表
        k: 团的大小

    输出:
        clauses: CNF 子句列表
        var_map: 变量映射
    """
    n = len(vertices)
    clauses = []

    # 变量: x_v 表示顶点 v 在团中
    var_map = {v: i + 1 for i, v in enumerate(vertices)}

    # 创建边集合（用于快速查找）
    edge_set = set(edges) | set((v, u) for u, v in edges)

    # 约束1: 不相邻的顶点不能同时在团中
    for i, u in enumerate(vertices):
        for v in vertices[i+1:]:
            if (u, v) not in edge_set:
                # u 和 v 不相邻，不能同时选
                clauses.append([-var_map[u], -var_map[v]])

    # 约束2: 至少选择 k 个顶点
    # 使用组合: 任意 n-k+1 个顶点中至少选一个
    # 这保证了至少选 k 个
    for combo in combinations(vertices, n - k + 1):
        clause = [var_map[v] for v in combo]
        clauses.append(clause)

    return clauses, var_map

def decode_clique(model: List[int], var_map: Dict, vertices: List[int]) -> Set[int]:
    """从 SAT 解码出团"""
    if model is None:
        return None

    model_set = set(model)
    return {v for v in vertices if var_map[v] in model_set}

In [23]:
# 示例：在一个图中找 3-团
print("示例: 在图中寻找大小为 3 的团")
print("\n原问题:")
print("  顶点: {0, 1, 2, 3, 4}")
print("  边: {(0,1), (0,2), (1,2), (1,3), (2,3), (3,4)}")
print("""
  图的结构:
      0---1---3---4
       \ / \ /
        2---+
  """)
print("  问题: 是否存在 3 个顶点两两相连?")

vertices_c = [0, 1, 2, 3, 4]
edges_c = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 3), (3, 4)]

clauses_c, var_map_c = clique_to_sat(vertices_c, edges_c, k=3)

print("\n变量定义:")
for v, var_id in var_map_c.items():
    print(f"  x_{var_id} = '顶点{v}在团中'")

print_cnf_info(clauses_c, "生成的 SAT 公式")

if HAS_PYSAT:
    sat, model = solve_sat(clauses_c)
    print(f"\n求解结果: {'可满足' if sat else '不可满足'}")
    if sat:
        clique = decode_clique(model, var_map_c, vertices_c)
        print(f"找到的团: {clique}")

        # 验证
        edge_set = set(edges_c) | set((v, u) for u, v in edges_c)
        clique_list = list(clique)
        print("验证 (团中顶点两两相连):")
        for i in range(len(clique_list)):
            for j in range(i+1, len(clique_list)):
                u, v = clique_list[i], clique_list[j]
                connected = (u, v) in edge_set
                print(f"  ({u}, {v}): {'相连 ✓' if connected else '不相连 ✗'}")

示例: 在图中寻找大小为 3 的团

原问题:
  顶点: {0, 1, 2, 3, 4}
  边: {(0,1), (0,2), (1,2), (1,3), (2,3), (3,4)}
  
  图的结构:
      0---1---3---4
       \ / \ /
        2---+
  
  问题: 是否存在 3 个顶点两两相连?

变量定义:
  x_1 = '顶点0在团中'
  x_2 = '顶点1在团中'
  x_3 = '顶点2在团中'
  x_4 = '顶点3在团中'
  x_5 = '顶点4在团中'

生成的 SAT 公式
变量数: 5
子句数: 14

DIMACS CNF 格式:
----------------------------------------
p cnf 5 14
-1 -4 0
-1 -5 0
-2 -5 0
-3 -5 0
1 2 3 0
1 2 4 0
1 2 5 0
1 3 4 0
1 3 5 0
1 4 5 0
2 3 4 0
2 3 5 0
2 4 5 0
3 4 5 0

求解结果: 可满足
找到的团: {0, 1, 2}
验证 (团中顶点两两相连):
  (0, 1): 相连 ✓
  (0, 2): 相连 ✓
  (1, 2): 相连 ✓


<>:9: SyntaxWarning: invalid escape sequence '\ '
<>:9: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipython-input-1118976676.py:9: SyntaxWarning: invalid escape sequence '\ '
  \ / \ /


---
## 4. 顶点覆盖 (Vertex Cover) → SAT

### 4.1 问题定义

**k-顶点覆盖问题**：给定图 $G = (V, E)$，判断是否存在最多 $k$ 个顶点的集合 $S$，使得每条边至少有一个端点在 $S$ 中。

### 4.2 规约方法

**变量定义**：$x_v$ 表示「顶点 $v$ 在覆盖集中」

**约束条件**：

1. **每条边至少有一个端点被选中**：
   $$\forall (u,v) \in E: (x_u \lor x_v)$$

2. **最多选择 k 个顶点**：使用基数约束
   - 任意 $k+1$ 个顶点中至少有一个不选
   $$\forall S \subseteq V, |S| = k+1: \bigvee_{v \in S} \neg x_v$$

In [24]:
def vertex_cover_to_sat(vertices: List[int], edges: List[Tuple[int, int]], k: int) -> Tuple[List[List[int]], Dict]:
    """
    将 k-顶点覆盖问题规约到 SAT

    输入:
        vertices: 顶点列表
        edges: 边列表
        k: 覆盖集大小上限

    输出:
        clauses: CNF 子句列表
        var_map: 变量映射
    """
    clauses = []

    # 变量: x_v 表示顶点 v 在覆盖集中
    var_map = {v: i + 1 for i, v in enumerate(vertices)}

    # 约束1: 每条边至少有一个端点被选中
    for (u, v) in edges:
        clauses.append([var_map[u], var_map[v]])

    # 约束2: 最多选择 k 个顶点
    # 任意 k+1 个顶点中至少有一个不选
    if k < len(vertices):
        for combo in combinations(vertices, k + 1):
            clause = [-var_map[v] for v in combo]
            clauses.append(clause)

    return clauses, var_map

def decode_vertex_cover(model: List[int], var_map: Dict, vertices: List[int]) -> Set[int]:
    """从 SAT 解码出顶点覆盖"""
    if model is None:
        return None

    model_set = set(model)
    return {v for v in vertices if var_map[v] in model_set}

In [25]:
# 示例：找最小顶点覆盖
print("示例: 寻找大小为 2 的顶点覆盖")
print("\n原问题:")
print("  顶点: {0, 1, 2, 3}")
print("  边: {(0,1), (1,2), (2,3)}")
print("""
  图的结构 (路径图):
      0---1---2---3
  """)
print("  问题: 能否用 2 个顶点覆盖所有边?")

vertices_vc = [0, 1, 2, 3]
edges_vc = [(0, 1), (1, 2), (2, 3)]

clauses_vc, var_map_vc = vertex_cover_to_sat(vertices_vc, edges_vc, k=2)

print("\n变量定义:")
for v, var_id in var_map_vc.items():
    print(f"  x_{var_id} = '顶点{v}在覆盖集中'")

print_cnf_info(clauses_vc, "生成的 SAT 公式")

if HAS_PYSAT:
    sat, model = solve_sat(clauses_vc)
    print(f"\n求解结果: {'可满足' if sat else '不可满足'}")
    if sat:
        cover = decode_vertex_cover(model, var_map_vc, vertices_vc)
        print(f"顶点覆盖: {cover}")

        # 验证
        print("验证 (每条边至少一个端点被覆盖):")
        for (u, v) in edges_vc:
            covered = u in cover or v in cover
            u_mark = "*" if u in cover else " "
            v_mark = "*" if v in cover else " "
            print(f"  边({u}{u_mark}, {v}{v_mark}): {'覆盖 ✓' if covered else '未覆盖 ✗'}")

示例: 寻找大小为 2 的顶点覆盖

原问题:
  顶点: {0, 1, 2, 3}
  边: {(0,1), (1,2), (2,3)}
  
  图的结构 (路径图):
      0---1---2---3
  
  问题: 能否用 2 个顶点覆盖所有边?

变量定义:
  x_1 = '顶点0在覆盖集中'
  x_2 = '顶点1在覆盖集中'
  x_3 = '顶点2在覆盖集中'
  x_4 = '顶点3在覆盖集中'

生成的 SAT 公式
变量数: 4
子句数: 7

DIMACS CNF 格式:
----------------------------------------
p cnf 4 7
1 2 0
2 3 0
3 4 0
-1 -2 -3 0
-1 -2 -4 0
-1 -3 -4 0
-2 -3 -4 0

求解结果: 可满足
顶点覆盖: {0, 2}
验证 (每条边至少一个端点被覆盖):
  边(0*, 1 ): 覆盖 ✓
  边(1 , 2*): 覆盖 ✓
  边(2*, 3 ): 覆盖 ✓


---
## 5. 哈密顿路径 (Hamiltonian Path) → SAT

### 5.1 问题定义

**哈密顿路径问题**：给定图 $G = (V, E)$，判断是否存在一条路径恰好经过每个顶点一次。

### 5.2 规约方法

**变量定义**：$x_{v,i}$ 表示「顶点 $v$ 是路径中的第 $i$ 个」

**约束条件**：

1. **每个顶点恰好出现在路径的某个位置**：
   - 至少出现: $\forall v: \bigvee_i x_{v,i}$
   - 至多一次: $\forall v, i < j: (\neg x_{v,i} \lor \neg x_{v,j})$

2. **每个位置恰好有一个顶点**：
   - 至少一个: $\forall i: \bigvee_v x_{v,i}$
   - 至多一个: $\forall i, u \neq v: (\neg x_{u,i} \lor \neg x_{v,i})$

3. **路径上相邻位置的顶点必须有边相连**：
   $$\forall i, \forall (u,v) \notin E: (\neg x_{u,i} \lor \neg x_{v,i+1})$$

In [26]:
def hamiltonian_path_to_sat(vertices: List[int], edges: List[Tuple[int, int]]) -> Tuple[List[List[int]], Dict]:
    """
    将哈密顿路径问题规约到 SAT

    输入:
        vertices: 顶点列表
        edges: 边列表

    输出:
        clauses: CNF 子句列表
        var_map: 变量映射 {(v, i): var_id}
    """
    n = len(vertices)
    clauses = []
    var_map = {}  # (vertex, position) -> variable_id

    # 创建变量映射
    var_id = 1
    for v in vertices:
        for i in range(n):
            var_map[(v, i)] = var_id
            var_id += 1

    # 创建边集合
    edge_set = set(edges) | set((v, u) for u, v in edges)

    # 约束1: 每个顶点至少在某个位置
    for v in vertices:
        clause = [var_map[(v, i)] for i in range(n)]
        clauses.append(clause)

    # 约束2: 每个顶点最多在一个位置
    for v in vertices:
        for i in range(n):
            for j in range(i + 1, n):
                clauses.append([-var_map[(v, i)], -var_map[(v, j)]])

    # 约束3: 每个位置至少有一个顶点
    for i in range(n):
        clause = [var_map[(v, i)] for v in vertices]
        clauses.append(clause)

    # 约束4: 每个位置最多有一个顶点
    for i in range(n):
        for j, u in enumerate(vertices):
            for v in vertices[j + 1:]:
                clauses.append([-var_map[(u, i)], -var_map[(v, i)]])

    # 约束5: 相邻位置的顶点必须有边
    for i in range(n - 1):
        for u in vertices:
            for v in vertices:
                if u != v and (u, v) not in edge_set:
                    clauses.append([-var_map[(u, i)], -var_map[(v, i + 1)]])

    return clauses, var_map

def decode_hamiltonian_path(model: List[int], var_map: Dict, vertices: List[int]) -> List[int]:
    """从 SAT 解码出哈密顿路径"""
    if model is None:
        return None

    n = len(vertices)
    model_set = set(model)
    path = [None] * n

    for v in vertices:
        for i in range(n):
            if var_map[(v, i)] in model_set:
                path[i] = v
                break

    return path

In [27]:
# 示例：在图中寻找哈密顿路径
print("示例: 寻找哈密顿路径")
print("\n原问题:")
print("  顶点: {0, 1, 2, 3}")
print("  边: {(0,1), (1,2), (2,3), (0,2)}")
print("""
  图的结构:
      0---1
      |\  |
      | \ |
      |  \|
      3---2
  """)
print("  问题: 是否存在经过每个顶点恰好一次的路径?")

vertices_hp = [0, 1, 2, 3]
edges_hp = [(0, 1), (1, 2), (2, 3), (0, 2)]

clauses_hp, var_map_hp = hamiltonian_path_to_sat(vertices_hp, edges_hp)

print(f"\n变量数: {len(var_map_hp)}")
print("变量定义示例:")
for i, ((v, pos), var_id) in enumerate(var_map_hp.items()):
    if i < 8:  # 只打印前几个
        print(f"  x_{var_id} = '顶点{v}在位置{pos}'")
print("  ...")

print(f"\n子句数: {len(clauses_hp)}")

if HAS_PYSAT:
    sat, model = solve_sat(clauses_hp)
    print(f"\n求解结果: {'可满足' if sat else '不可满足'}")
    if sat:
        path = decode_hamiltonian_path(model, var_map_hp, vertices_hp)
        print(f"哈密顿路径: {' → '.join(map(str, path))}")

        # 验证
        edge_set = set(edges_hp) | set((v, u) for u, v in edges_hp)
        print("验证 (相邻顶点有边连接):")
        for i in range(len(path) - 1):
            u, v = path[i], path[i + 1]
            connected = (u, v) in edge_set
            print(f"  {u} → {v}: {'有边 ✓' if connected else '无边 ✗'}")

示例: 寻找哈密顿路径

原问题:
  顶点: {0, 1, 2, 3}
  边: {(0,1), (1,2), (2,3), (0,2)}
  
  图的结构:
      0---1
      |\  |
      | \ |
      |  \|
      3---2
  
  问题: 是否存在经过每个顶点恰好一次的路径?

变量数: 16
变量定义示例:
  x_1 = '顶点0在位置0'
  x_2 = '顶点0在位置1'
  x_3 = '顶点0在位置2'
  x_4 = '顶点0在位置3'
  x_5 = '顶点1在位置0'
  x_6 = '顶点1在位置1'
  x_7 = '顶点1在位置2'
  x_8 = '顶点1在位置3'
  ...

子句数: 68

求解结果: 可满足
哈密顿路径: 3 → 2 → 1 → 0
验证 (相邻顶点有边连接):
  3 → 2: 有边 ✓
  2 → 1: 有边 ✓
  1 → 0: 有边 ✓


<>:9: SyntaxWarning: invalid escape sequence '\ '
<>:9: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipython-input-343569262.py:9: SyntaxWarning: invalid escape sequence '\ '
  |\  |


In [28]:
# 示例：一个没有哈密顿路径的图
print("\n" + "="*60)
print("示例: 没有哈密顿路径的图")
print("="*60)
print("\n原问题:")
print("  顶点: {0, 1, 2, 3}")
print("  边: {(0,1), (2,3)} (两个不连通的边)")
print("""
  图的结构:
      0---1     2---3
  """)

vertices_no_hp = [0, 1, 2, 3]
edges_no_hp = [(0, 1), (2, 3)]

clauses_no_hp, var_map_no_hp = hamiltonian_path_to_sat(vertices_no_hp, edges_no_hp)

if HAS_PYSAT:
    sat, model = solve_sat(clauses_no_hp)
    print(f"\n求解结果: {'可满足' if sat else '不可满足'}")
    if not sat:
        print("说明: 图不连通，无法找到哈密顿路径")


示例: 没有哈密顿路径的图

原问题:
  顶点: {0, 1, 2, 3}
  边: {(0,1), (2,3)} (两个不连通的边)
  
  图的结构:
      0---1     2---3
  

求解结果: 不可满足
说明: 图不连通，无法找到哈密顿路径


---
## 6. 子集和 (Subset Sum) → SAT

### 6.1 问题定义

**子集和问题**：给定一组正整数 $S = \{a_1, a_2, \ldots, a_n\}$ 和目标值 $T$，判断是否存在子集使得其和恰好为 $T$。

### 6.2 规约方法

这个规约比较复杂，需要用二进制表示来编码加法。我们使用一种简化的方法：

**变量定义**：
- $x_i$ 表示「元素 $a_i$ 被选中」
- $s_{j,b}$ 表示「前 $j$ 个元素的部分和的第 $b$ 位是 1」

这需要实现二进制加法器的逻辑。

In [29]:
def subset_sum_to_sat_simple(numbers: List[int], target: int) -> Tuple[List[List[int]], Dict]:
    """
    将子集和问题规约到 SAT（简化版本，使用辅助变量）

    这个版本使用 (i, s) 表示「考虑前 i 个元素能否得到和 s」
    变量 reach[i][s] 表示用前 i 个元素能否达到和 s

    注意：这种方法对于大的 target 会产生很多变量
    """
    n = len(numbers)
    clauses = []
    var_map = {}  # 包含选择变量和到达变量

    var_id = 1

    # 选择变量: x_i 表示选择第 i 个数
    select_vars = {}
    for i in range(n):
        select_vars[i] = var_id
        var_map[f'select_{i}'] = var_id
        var_id += 1

    # 可达变量: reach[i][s] 表示用前 i 个数能否达到和 s
    # 为了避免太多变量，我们只考虑 0 到 target 的和
    reach_vars = {}  # (i, s) -> var_id
    for i in range(n + 1):
        for s in range(target + 1):
            reach_vars[(i, s)] = var_id
            var_map[f'reach_{i}_{s}'] = var_id
            var_id += 1

    # 初始条件: 前 0 个数只能达到和 0
    clauses.append([reach_vars[(0, 0)]])  # reach[0][0] = True
    for s in range(1, target + 1):
        clauses.append([-reach_vars[(0, s)]])  # reach[0][s] = False for s > 0

    # 转移关系
    for i in range(n):
        a = numbers[i]
        for s in range(target + 1):
            # reach[i+1][s] = reach[i][s] OR (select[i] AND reach[i][s-a])
            # 转换为 CNF:
            # reach[i+1][s] -> reach[i][s] OR (select[i] AND reach[i][s-a])
            # NOT reach[i+1][s] OR reach[i][s] OR (select[i] AND reach[i][s-a])

            # 如果 reach[i][s] 为真，则 reach[i+1][s] 为真
            clauses.append([-reach_vars[(i, s)], reach_vars[(i + 1, s)]])

            # 如果选择 i 且 reach[i][s-a] 为真，则 reach[i+1][s] 为真
            if s >= a:
                clauses.append([-select_vars[i], -reach_vars[(i, s - a)], reach_vars[(i + 1, s)]])

            # 反向: 如果 reach[i+1][s] 为真，则必须是上述两种情况之一
            # reach[i+1][s] -> reach[i][s] OR (select[i] AND reach[i][s-a])
            if s >= a:
                clauses.append([-reach_vars[(i + 1, s)], reach_vars[(i, s)], select_vars[i]])
                clauses.append([-reach_vars[(i + 1, s)], reach_vars[(i, s)], reach_vars[(i, s - a)]])
            else:
                clauses.append([-reach_vars[(i + 1, s)], reach_vars[(i, s)]])

    # 目标: reach[n][target] 必须为真
    clauses.append([reach_vars[(n, target)]])

    return clauses, {'select': select_vars, 'reach': reach_vars}

def decode_subset_sum(model: List[int], var_info: Dict, numbers: List[int]) -> List[int]:
    """从 SAT 解码出选择的子集"""
    if model is None:
        return None

    model_set = set(model)
    select_vars = var_info['select']

    selected = [numbers[i] for i in range(len(numbers)) if select_vars[i] in model_set]
    return selected

In [30]:
# 示例：子集和问题
print("示例: 子集和问题")
print("\n原问题:")
print("  集合: {3, 7, 1, 8, 4}")
print("  目标: 11")
print("  问题: 是否存在子集使得和为 11?")

numbers = [3, 7, 1, 8, 4]
target = 11

clauses_ss, var_info_ss = subset_sum_to_sat_simple(numbers, target)

print(f"\n变量数: {max(abs(lit) for clause in clauses_ss for lit in clause)}")
print(f"子句数: {len(clauses_ss)}")

print("\n选择变量:")
for i, var_id in var_info_ss['select'].items():
    print(f"  x_{var_id} = '选择数字 {numbers[i]}'")

if HAS_PYSAT:
    sat, model = solve_sat(clauses_ss)
    print(f"\n求解结果: {'可满足' if sat else '不可满足'}")
    if sat:
        subset = decode_subset_sum(model, var_info_ss, numbers)
        print(f"选择的子集: {subset}")
        print(f"验证: {' + '.join(map(str, subset))} = {sum(subset)}")

示例: 子集和问题

原问题:
  集合: {3, 7, 1, 8, 4}
  目标: 11
  问题: 是否存在子集使得和为 11?

变量数: 77
子句数: 207

选择变量:
  x_1 = '选择数字 3'
  x_2 = '选择数字 7'
  x_3 = '选择数字 1'
  x_4 = '选择数字 8'
  x_5 = '选择数字 4'

求解结果: 可满足
选择的子集: [7, 4]
验证: 7 + 4 = 11


In [15]:
# 示例：不可满足的子集和
print("\n" + "="*60)
print("示例: 不可满足的子集和")
print("="*60)
print("\n原问题:")
print("  集合: {2, 4, 6, 8}")
print("  目标: 7 (奇数)")
print("  问题: 是否存在子集使得和为 7?")
print("  注: 所有数都是偶数，不可能得到奇数和")

numbers_no = [2, 4, 6, 8]
target_no = 7

clauses_no, var_info_no = subset_sum_to_sat_simple(numbers_no, target_no)

if HAS_PYSAT:
    sat, model = solve_sat(clauses_no)
    print(f"\n求解结果: {'可满足' if sat else '不可满足'}")
    if not sat:
        print("说明: 偶数之和不可能为奇数")


示例: 不可满足的子集和

原问题:
  集合: {2, 4, 6, 8}
  目标: 7 (奇数)
  问题: 是否存在子集使得和为 7?
  注: 所有数都是偶数，不可能得到奇数和


---
## 总结

### 规约的核心思想

所有 NP-完全问题都可以规约到 SAT，关键步骤是：

1. **定义布尔变量**：用变量表示原问题的决策
2. **编码约束**：将原问题的约束转换为 CNF 子句
3. **保持等价性**：确保 SAT 可满足 ⟺ 原问题有解

### 各问题的规约复杂度

| 问题 | 变量数 | 子句数 | 说明 |
|------|--------|--------|------|
| k-图着色 | O(nk) | O(nk² + mk) | n=顶点数, m=边数 |
| k-团 | O(n) | O(n²) + O(C(n, n-k+1)) | 基数约束较多 |
| k-顶点覆盖 | O(n) | O(m) + O(C(n, k+1)) | 基数约束较多 |
| 哈密顿路径 | O(n²) | O(n³) | 位置×顶点编码 |
| 子集和 | O(n·T) | O(n·T) | T=目标值 |

### Cook-Levin 定理的意义

SAT 是第一个被证明为 NP-完全的问题（Cook, 1971; Levin, 1973）。

这意味着：
- 如果能在多项式时间内解决 SAT，就能解决所有 NP 问题
- SAT 求解器的进步对整个 NP 问题类都有帮助
- 现代 SAT 求解器（如 MiniSat, Glucose）已经非常高效

---
## 练习

1. **3-SAT 到图着色**：尝试将一个 3-SAT 实例规约到 3-图着色问题。

2. **独立集到顶点覆盖**：证明 $S$ 是独立集 ⟺ $V \setminus S$ 是顶点覆盖。

3. **优化版本**：修改上面的代码，让它支持寻找最小顶点覆盖（用二分搜索）。

4. **哈密顿回路**：修改哈密顿路径的规约，使其寻找哈密顿回路（首尾相连）。